# Task 6: House Price Prediction

## Objective
Predict house prices using property features such as size, bedrooms, and location using regression models.

## Approach
- Perform preprocessing on features like square footage, number of bedrooms, location
- Train regression models (Linear Regression & Gradient Boosting)
- Evaluate with Mean Absolute Error (MAE) and RMSE
- Visualize predicted vs actual prices

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print("All libraries imported successfully!")

## 2. Load Dataset
We use the California Housing dataset (from sklearn) as a proxy - it contains median house values based on location, rooms, and demographics. Similar features to a standard house price dataset.

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame

df.rename(columns={
    'MedInc': 'median_income',
    'HouseAge': 'house_age',
    'AveRooms': 'avg_rooms',
    'AveBedrms': 'avg_bedrooms',
    'Population': 'population',
    'AveOccup': 'avg_occupancy',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'MedHouseVal': 'house_value'
}, inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget: Median house value ($100,000s)")
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Info:")
df.info()
print("\nDescriptive Statistics:")
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

features = ['median_income', 'house_age', 'avg_rooms', 'avg_bedrooms', 'population', 'avg_occupancy', 'latitude', 'longitude']
for i, feat in enumerate(features):
    row, col = i // 3, i % 3
    axes[row, col].scatter(df[feat], df['house_value'], alpha=0.3, s=5)
    axes[row, col].set_xlabel(feat)
    axes[row, col].set_ylabel('House Value ($100k)')
    axes[row, col].set_title(f'{feat} vs House Value')
    axes[row, col].grid(True, alpha=0.3)

# Hide extra subplot
axes[2, 2].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['house_value'], kde=True, bins=50, ax=axes[0])
axes[0].set_title('Distribution of House Values')
axes[0].set_xlabel('House Value ($100k)')

sns.boxplot(data=df, y='house_value', ax=axes[1])
axes[1].set_title('Box Plot of House Values')

sns.scatterplot(data=df, x='median_income', y='house_value', alpha=0.3, ax=axes[2])
axes[2].set_title('Median Income vs House Value')
axes[2].set_xlabel('Median Income ($10k)')

plt.tight_layout()
plt.show()

print(f"Skewness of house_value: {df['house_value'].skew():.3f}")

In [ ]:
# Correlation analysis
plt.figure(figsize=(12, 10))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nTop features correlated with house value:")
print(correlation['house_value'].abs().sort_values(ascending=False))

## 4. Feature Engineering

In [ ]:
# Feature engineering - create new features
df['rooms_per_household'] = df['avg_rooms'] / df['avg_occupancy']
df['bedrooms_per_room'] = df['avg_bedrooms'] / df['avg_rooms']
df['population_per_household'] = df['population'] / df['avg_occupancy']

# Feature: Location-based cluster (simplified)
df['location_score'] = (df['latitude'] * df['longitude']).abs()

print("New features created!")
df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household', 'location_score']].head()

In [ ]:
# Separate features and target
X = df.drop('house_value', axis=1)
y = df['house_value']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training

### 5.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"MAE:  ${mae_lr * 100000:.2f}")
print(f"RMSE: ${rmse_lr * 100000:.2f}")
print(f"R² Score: {r2_lr:.4f}")
print(f"R² Score: {r2_lr:.4f}")

### 5.2 Gradient Boosting Regressor

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test)

mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
rmse_gbr = np.sqrt(mean_squared_error(y_test, y_pred_gbr))
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Results:")
print(f"MAE:  ${mae_gbr * 100000:.2f}")
print(f"RMSE: ${rmse_gbr * 100000:.2f}")
print(f"R² Score: {r2_gbr:.4f}")

## 6. Model Evaluation

### 6.1 Predicted vs Actual Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, y_pred) in zip(axes, [('Linear Regression', y_pred_lr), ('Gradient Boosting', y_pred_gbr)]):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual House Value ($100k)')
    ax.set_ylabel('Predicted House Value ($100k)')
    ax.set_title(f'{name}: Predicted vs Actual')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

residuals_lr = y_test - y_pred_lr
residuals_gbr = y_test - y_pred_gbr

for ax, (name, residuals, y_pred) in zip(axes, 
    [('Linear Regression', residuals_lr, y_pred_lr), 
     ('Gradient Boosting', residuals_gbr, y_pred_gbr)]):
    ax.scatter(y_pred, residuals, alpha=0.3, s=10)
    ax.axhline(y=0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Predicted Values ($100k)')
    ax.set_ylabel('Residuals ($100k)')
    ax.set_title(f'{name}: Residual Plot')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.3 Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (name, residuals) in zip(axes, [('Linear Regression', residuals_lr), ('Gradient Boosting', residuals_gbr)]):
    sns.histplot(residuals, kde=True, bins=50, ax=ax)
    ax.axvline(x=0, color='r', linestyle='--', lw=2)
    ax.set_xlabel('Prediction Error ($100k)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name}: Error Distribution')

plt.tight_layout()
plt.show()

### 6.4 Feature Importance (from Gradient Boosting)

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': gbr.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis_r')
plt.title('Gradient Boosting - Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.head(5).to_string(index=False))

## 7. Cross-Validation

In [ ]:
# Cross-validation scores
cv_scores_lr = cross_val_score(LinearRegression(), X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_gbr = cross_val_score(GradientBoostingRegressor(n_estimators=100, random_state=42), X_train, y_train, cv=3, scoring='r2')

print(f"Linear Regression - CV R²: {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")
print(f"Gradient Boosting - CV R²: {cv_scores_gbr.mean():.4f} (+/- {cv_scores_gbr.std():.4f})")

## 8. Results Summary

### Key Findings:
1. **Best Model**: Gradient Boosting Regressor significantly outperforms Linear Regression
2. **Most Important Features**:
   - `median_income`: Strongest predictor of house value
   - `location_score` (latitude × longitude): Geographic location matters
   - `avg_occupancy`: Average household size impact
   - `avg_rooms`: Property size indicator

3. **Model Performance**:
   - Gradient Boosting: Lower MAE and RMSE, higher R²
   - Linear Regression: Simpler, faster but less accurate

### Conclusion:
The Gradient Boosting model provides the most accurate house price predictions. Median income and location are the dominant factors in determining house prices in California.

In [ ]:
# Final comparison
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Gradient Boosting'],
    'MAE': [mae_lr, mae_gbr],
    'RMSE': [rmse_lr, rmse_gbr],
    'R²': [r2_lr, r2_gbr]
})

results['MAE'] = results['MAE'].apply(lambda x: f'${x*100000:.2f}')
results['RMSE'] = results['RMSE'].apply(lambda x: f'${x*100000:.2f}')
print(results.to_string(index=False))
print("\nTask 6 completed successfully!")